# Crop Recommendation Model Comparison

This notebook is designed for Google Colab. It builds a multiclass crop recommendation system that predicts `Crop_Type` with confidence scores for each crop.

Models compared:
- ANN
- RandomForest
- XGBoost if available, otherwise GradientBoosting as fallback

The dataset contains `19` crop types.


## 1. Optional Install for Colab

In [ ]:
# Uncomment in Colab if needed
# !pip install xgboost

## 2. Imports and Setup

In [ ]:
import json
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss, top_k_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('XGBoost available:', XGBOOST_AVAILABLE)


## 3. Load Dataset

In [ ]:
PROJECT_ROOT = Path(r'C:\Users\manis\OneDrive\Desktop\PA-project')
CSV_PATH = PROJECT_ROOT / 'Dataset' / 'solit_dataset' / 'Soil-Climate-data.csv'
MODELS_DIR = PROJECT_ROOT / 'Models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print('Rows:', len(df))
print('Crop types:', df['Crop_Type'].nunique())
display(df.head())


In [ ]:
crop_counts = df['Crop_Type'].value_counts().sort_values(ascending=False)
crop_counts


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x=crop_counts.index, y=crop_counts.values)
plt.title('Crop Type Distribution')
plt.xlabel('Crop Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 4. Feature Design

Target:
- `Crop_Type`

Input features:
- categorical: `Soil_Type`, `Irrigation_Available`
- numeric: `Farm_Size_Acres`, `Soil_pH`, `Soil_Nitrogen`, `Soil_Organic_Matter`, `Temperature`, `Rainfall`, `Humidity`, `Compatible`

`Crop_Type` is excluded from the features because it is the label.


In [ ]:
target_col = 'Crop_Type'
categorical_features = ['Soil_Type', 'Irrigation_Available']
numeric_features = [
    'Farm_Size_Acres',
    'Soil_pH',
    'Soil_Nitrogen',
    'Soil_Organic_Matter',
    'Temperature',
    'Rainfall',
    'Humidity',
    'Compatible',
]

X = df[categorical_features + numeric_features].copy()
y = df[target_col].copy()

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = list(label_encoder.classes_)
print('Number of crop types:', len(class_names))
print(class_names)


## 5. Stratified Train/Validation/Test Split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    random_state=SEED,
    stratify=y_encoded,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print('Train:', len(X_train))
print('Validation:', len(X_val))
print('Test:', len(X_test))


## 6. Preprocessing

In [ ]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), numeric_features),
    ]
)

ann_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
    ]
)

X_train_tree = tree_preprocessor.fit_transform(X_train)
X_val_tree = tree_preprocessor.transform(X_val)
X_test_tree = tree_preprocessor.transform(X_test)

X_train_ann = ann_preprocessor.fit_transform(X_train)
X_val_ann = ann_preprocessor.transform(X_val)
X_test_ann = ann_preprocessor.transform(X_test)

if hasattr(X_train_ann, 'toarray'):
    X_train_ann = X_train_ann.toarray()
    X_val_ann = X_val_ann.toarray()
    X_test_ann = X_test_ann.toarray()

print('ANN input shape:', X_train_ann.shape)
print('Tree input shape:', X_train_tree.shape)


## 7. ANN Model

In [ ]:
class CropANN(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


train_ds = TensorDataset(torch.tensor(X_train_ann, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val_ann, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(X_test_ann, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)


def evaluate_ann(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    preds = []
    targets = []
    probs = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
    return running_loss / len(loader.dataset), np.array(targets), np.array(preds), np.array(probs)


def train_ann(epochs=40, lr=1e-3):
    model = CropANN(X_train_ann.shape[1], len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_epoch = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            correct += outputs.argmax(dim=1).eq(labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, y_val_true, y_val_pred, y_val_prob = evaluate_ann(model, val_loader, criterion)
        val_acc = accuracy_score(y_val_true, y_val_pred)

        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'ANN | Epoch {epoch + 1}/{epochs} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_loss, y_true, y_pred, y_prob = evaluate_ann(model, test_loader, criterion)
    return {
        'model_name': 'ANN',
        'model': model,
        'history': pd.DataFrame(history),
        'best_epoch': best_epoch,
        'best_val_acc': best_val_acc,
        'test_acc': accuracy_score(y_true, y_pred),
        'test_log_loss': log_loss(y_true, y_prob),
        'top3_acc': top_k_accuracy_score(y_true, y_prob, k=3, labels=np.arange(len(class_names))),
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }


ann_result = train_ann()


## 8. Tree-Based Models

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    random_state=SEED,
    n_jobs=-1,
)
rf_model.fit(X_train_tree, y_train)
rf_val_prob = rf_model.predict_proba(X_val_tree)
rf_test_prob = rf_model.predict_proba(X_test_tree)
rf_result = {
    'model_name': 'RandomForest',
    'model': rf_model,
    'best_epoch': None,
    'best_val_acc': accuracy_score(y_val, rf_val_prob.argmax(axis=1)),
    'test_acc': accuracy_score(y_test, rf_test_prob.argmax(axis=1)),
    'test_log_loss': log_loss(y_test, rf_test_prob),
    'top3_acc': top_k_accuracy_score(y_test, rf_test_prob, k=3, labels=np.arange(len(class_names))),
    'y_true': np.array(y_test),
    'y_pred': rf_test_prob.argmax(axis=1),
    'y_prob': rf_test_prob,
}

if XGBOOST_AVAILABLE:
    booster = XGBClassifier(
        objective='multi:softprob',
        num_class=len(class_names),
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='mlogloss',
        random_state=SEED,
    )
    booster_name = 'XGBoost'
else:
    booster = GradientBoostingClassifier(random_state=SEED)
    booster_name = 'GradientBoosting'

booster.fit(X_train_tree, y_train)
booster_val_prob = booster.predict_proba(X_val_tree)
booster_test_prob = booster.predict_proba(X_test_tree)
boost_result = {
    'model_name': booster_name,
    'model': booster,
    'best_epoch': None,
    'best_val_acc': accuracy_score(y_val, booster_val_prob.argmax(axis=1)),
    'test_acc': accuracy_score(y_test, booster_test_prob.argmax(axis=1)),
    'test_log_loss': log_loss(y_test, booster_test_prob),
    'top3_acc': top_k_accuracy_score(y_test, booster_test_prob, k=3, labels=np.arange(len(class_names))),
    'y_true': np.array(y_test),
    'y_pred': booster_test_prob.argmax(axis=1),
    'y_prob': booster_test_prob,
}

model_results = [ann_result, rf_result, boost_result]


## 9. Comparison Table

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': result['model_name'],
        'Best Validation Accuracy': result['best_val_acc'],
        'Test Accuracy': result['test_acc'],
        'Top-3 Accuracy': result['top3_acc'],
        'Test Log Loss': result['test_log_loss'],
    }
    for result in model_results
]).sort_values('Test Accuracy', ascending=False)
comparison_df


In [ ]:
plt.figure(figsize=(9, 5))
plot_df = comparison_df.melt(id_vars='Model', value_vars=['Best Validation Accuracy', 'Test Accuracy', 'Top-3 Accuracy'])
sns.barplot(data=plot_df, x='Model', y='value', hue='variable')
plt.ylim(0, 1)
plt.title('Crop Recommendation Model Comparison')
plt.ylabel('Score')
plt.show()


## 10. ANN Learning Curves

In [ ]:
ann_history = ann_result['history']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ann_history['epoch'], ann_history['train_acc'], label='Train Accuracy')
axes[0].plot(ann_history['epoch'], ann_history['val_acc'], label='Validation Accuracy')
axes[0].set_title('ANN Accuracy Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(ann_history['epoch'], ann_history['train_loss'], label='Train Loss')
axes[1].plot(ann_history['epoch'], ann_history['val_loss'], label='Validation Loss')
axes[1].set_title('ANN Loss Curves')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
plt.tight_layout()
plt.show()


## 11. Confusion Matrix for Best Model

In [ ]:
best_result = max(model_results, key=lambda item: item['test_acc'])
cm = confusion_matrix(best_result['y_true'], best_result['y_pred'])
plt.figure(figsize=(14, 10))
sns.heatmap(cm, cmap='Blues')
plt.title(f"{best_result['model_name']} Test Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()
print(classification_report(best_result['y_true'], best_result['y_pred'], target_names=class_names))


## 12. Confidence Scores for a Single Input

In [ ]:
sample_row = X_test.iloc[[0]].copy()
sample_true_label = class_names[y_test[0]]
display(sample_row)
print('True crop:', sample_true_label)

if best_result['model_name'] == 'ANN':
    sample_features = ann_preprocessor.transform(sample_row)
    if hasattr(sample_features, 'toarray'):
        sample_features = sample_features.toarray()
    sample_tensor = torch.tensor(sample_features, dtype=torch.float32).to(device)
    with torch.no_grad():
        sample_prob = torch.softmax(best_result['model'](sample_tensor), dim=1).cpu().numpy()[0]
else:
    sample_features = tree_preprocessor.transform(sample_row)
    sample_prob = best_result['model'].predict_proba(sample_features)[0]

confidence_df = pd.DataFrame({
    'Crop_Type': class_names,
    'Confidence': sample_prob,
}).sort_values('Confidence', ascending=False)
confidence_df.head(10)


In [ ]:
plt.figure(figsize=(10, 5))
top10 = confidence_df.head(10)
sns.barplot(data=top10, x='Confidence', y='Crop_Type')
plt.title(f"Top Crop Recommendations from {best_result['model_name']}")
plt.xlim(0, 1)
plt.show()


## 13. Save Artifacts

In [ ]:
torch.save(ann_result['model'].state_dict(), MODELS_DIR / 'crop_ann_multiclass_model.pth')
joblib.dump(rf_result['model'], MODELS_DIR / 'crop_random_forest_model.pkl')
joblib.dump(boost_result['model'], MODELS_DIR / 'crop_boosting_model.pkl')
joblib.dump(label_encoder, MODELS_DIR / 'crop_label_encoder.pkl')
joblib.dump(tree_preprocessor, MODELS_DIR / 'crop_tree_preprocessor.pkl')
joblib.dump(ann_preprocessor, MODELS_DIR / 'crop_ann_preprocessor.pkl')

metrics_payload = {
    'task': 'crop recommendation',
    'crop_type_count': len(class_names),
    'crop_types': class_names,
    'train_size': int(len(X_train)),
    'validation_size': int(len(X_val)),
    'test_size': int(len(X_test)),
    'models': comparison_df.to_dict(orient='records'),
    'best_model': comparison_df.iloc[0]['Model'],
}

with open(MODELS_DIR / 'crop_recommendation_model_comparison.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2)

comparison_df.to_csv(MODELS_DIR / 'crop_recommendation_model_comparison.csv', index=False)
comparison_df
